# LC 503 — Next Greater Element II
**Day-71 | Monotonic Stack | Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> A circular array means every element
can "wrap around" to find a greater element after the end.
Simulate the wrap by iterating <em>twice</em> (indices 0 to 2n-1,
using <code>i % n</code>) with a monotonic <em>decreasing</em>
stack of indices. Only push indices during the first pass
(i &lt; n) to avoid double-counting.
</div>

## Official Problem Statement

Given a circular integer array `nums` (i.e., the next element of
`nums[nums.length - 1]` is `nums[0]`), return the **next greater
number** for every element in `nums`.

The next greater number of a number `x` is the first greater number
to its **traversing-order next** in the array, which means you could
search circularly to find its next greater number. If it doesn't
exist, return `-1` for this number.

**Example:**
```
Input:  nums = [1, 2, 1]
Output: [2, -1, 2]
```

**Constraints:**
- `1 <= nums.length <= 10^4`
- `-10^9 <= nums[i] <= 10^9`

## What This Is Actually Asking

For each position `i`, scan rightward (wrapping around if needed)
until you find a value strictly greater than `nums[i]`.
That value is the answer. If no such value exists even after a
full loop, return -1.

```
nums = [1, 2, 1]

index 0 (value=1): scan → 2 > 1 ✓  answer=2
index 1 (value=2): scan → 1, wrap → 1, 2 not > 2, back to start
                   no element > 2        answer=-1
index 2 (value=1): scan → wrap → 1, 2 > 1 ✓  answer=2
```

Naive: O(n²) — for each element, scan up to 2n elements.
Smart: O(n) — monotonic stack processes each element at most
twice (once per simulated pass).

## Walk Through an Example by Hand

`nums = [5, 4, 3, 2, 1]`  (n=5)
Expected: `[-1, 5, 5, 5, 5]`

```
result = [-1, -1, -1, -1, -1]   stack = []

i=0, val=nums[0]=5
  stack empty → push 0    stack=[0]

i=1, val=nums[1]=4
  nums[stack[-1]]=nums[0]=5 > 4 → no pop
  push 1                   stack=[0,1]

i=2, val=nums[2]=3  → push 2   stack=[0,1,2]
i=3, val=nums[3]=2  → push 3   stack=[0,1,2,3]
i=4, val=nums[4]=1  → push 4   stack=[0,1,2,3,4]

--- second pass (i=5..9, i<n is False so no push) ---

i=5, val=nums[5%5]=nums[0]=5
  pop 4: nums[4]=1 < 5 → result[4]=5  stack=[0,1,2,3]
  pop 3: nums[3]=2 < 5 → result[3]=5  stack=[0,1,2]
  pop 2: nums[2]=3 < 5 → result[2]=5  stack=[0,1]
  pop 1: nums[1]=4 < 5 → result[1]=5  stack=[0]
  nums[0]=5 not < 5 → stop
  i>=n → no push

i=6..9: val cycles 4,3,2,1 — all < stack top (5) → no pops

result = [-1, 5, 5, 5, 5]  ✓
```

## The Picture

Bar chart for `nums = [1, 2, 1]` (circular wrap shown by arrows):

```
  2  |    ##
  1  | ##      ##
      i=0  i=1  i=2  → wraps back to i=0

  i=0 (1): next greater → 2  (one step right)
  i=1 (2): next greater → ∅  (no element > 2 in full circle)
  i=2 (1): next greater → 1  (wrap) → 2 found at i=1
```

Stack trace (indices, decreasing by value):

```
Pass 1 (i = 0..n-1):
  i=0, val=1 → stack empty → push 0   stack=[0]
  i=1, val=2 → nums[0]=1<2 pop→res[0]=2 → stack=[] → push 1
                                           stack=[1]
  i=2, val=1 → nums[1]=2>1 stop → push 2  stack=[1,2]

Pass 2 (i = n..2n-1, no pushes):
  i=3 → i%n=0, val=1 → nums[1]=2>1 stop (no change)
  i=4 → i%n=1, val=2 → nums[1]=2 not < 2 stop
  i=5 → i%n=2, val=1 → nums[1]=2>1 stop

Remaining stack [1,2] → result stays -1 for those.
Final: [2, -1, 2]  ✓
```

Key: Stack is always **decreasing by value**. An incoming
larger value "resolves" all smaller waiting indices.

## When To Use This Pattern

Use **monotonic stack + double-pass** when:
- Array is circular (next element of last = first element)
- You need "next greater" or "next smaller" with wraparound
- A single linear scan can't see elements that come before
  the current position in the original array

**The double-pass trick generalises:**
- Concatenate the array with itself conceptually: length 2n
- Use `i % n` to index into the real array
- Only push onto the stack during the first n iterations
  to avoid re-registering indices

**Related problems:**
- LC 496 Next Greater Element I (non-circular, use hashmap)
- LC 739 Daily Temperatures (same pattern, store distances)
- LC 84 Largest Rectangle in Histogram (areas not elements)

## The Approach

**Data structure:** Stack of **indices** (monotonic decreasing
by `nums[index]` value).

**Algorithm:**
1. `n = len(nums)`
2. `result = [-1] * n`
3. `stack = []`  (stores indices)
4. Iterate `i` from `0` to `2n - 1`:
   - `val = nums[i % n]`
   - While `stack` is not empty AND `nums[stack[-1]] < val`:
     - `idx = stack.pop()`
     - `result[idx] = val`
   - If `i < n`: push `i` onto stack
5. Return `result`

**Why indices (not values)?** We need to write back to
`result[idx]`, so we must remember the original position.

**Why only push when `i < n`?** Each original index should
appear on the stack at most once. The second pass only
resolves pending indices, never adds new ones.

**Complexity:** O(n) time, O(n) space.

In [1]:
from typing import List

In [2]:
# ── Test harness ─────────────────────────────────────────────────────

def test_harness(func):
    """
    Runs func(nums) against known test cases.
    Prints PASSED / FAILED per case and a final summary.

    Args:
        func: callable(List[int]) -> List[int]
    """
    test_cases = [
        {
            "nums":   [1, 2, 1],
            "expect": [2, -1, 2],
            "label":  "Example 1 (LeetCode)",
        },
        {
            "nums":   [1, 2, 3, 4, 3],
            "expect": [2, 3, 4, -1, 4],
            "label":  "Example 2 (LeetCode)",
        },
        {
            "nums":   [5, 4, 3, 2, 1],
            "expect": [-1, 5, 5, 5, 5],
            "label":  "Strictly decreasing",
        },
        {
            "nums":   [1, 2, 3],
            "expect": [2, 3, -1],
            "label":  "Strictly increasing",
        },
        {
            "nums":   [3, 3, 3],
            "expect": [-1, -1, -1],
            "label":  "All equal",
        },
        {
            "nums":   [7],
            "expect": [-1],
            "label":  "Single element",
        },
    ]

    passed = 0
    for tc in test_cases:
        result = func(tc["nums"])
        ok = result == tc["expect"]
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(f"  [{status}] {tc['label']}")
        if not ok:
            print(f"    got:      {result}")
            print(f"    expected: {tc['expect']}")

    total = len(test_cases)
    print(f"\n  Summary: {passed}/{total} passed")


print("Test harness defined.")

Test harness defined.


In [10]:
def next_greater_elements(nums: List[int]) -> List[int]:
    n = len(nums)
    res = [-1] * n                 # pre-filled — leftovers stay -1
    stack: List[int] = []          # indices waiting for their NGE

    for i in range(2 * n):
        val = nums[i % n]
        while stack and nums[stack[-1]] < val:
            res[stack.pop()] = val  # write answer directly by index
        if i < n:                   # only push first pass — no duplicates
            stack.append(i)

    return res
print(next_greater_elements([1, 2, 3, 4, 3]))
test_harness(next_greater_elements)
print("next_greater_elements shell defined.")



[2, 3, 4, -1, 4]
  [PASSED] Example 1 (LeetCode)
  [PASSED] Example 2 (LeetCode)
  [PASSED] Strictly decreasing
  [PASSED] Strictly increasing
  [PASSED] All equal
  [PASSED] Single element

  Summary: 6/6 passed
next_greater_elements shell defined.


In [ ]:
# Uncomment and run when solution is ready
# test_harness(next_greater_elements)

## Complexity

| | Time | Space |
|---|---|---|
| Overall | **O(n)** | **O(n)** |

**Time breakdown:**
- The loop runs `2n` iterations.
- Each of the `n` original indices is pushed onto the stack
  at most once and popped at most once.
- Total stack operations ≤ 2n → O(n) overall.

**Space breakdown:**
- `result` array: O(n)
- `stack` worst case: O(n) — strictly decreasing array means
  all indices stay on stack until the second pass.

**Compared to brute force:**
- Brute force: O(n²) time, O(1) extra space (scan up to 2n
  positions for each of n elements).
- Stack approach: O(n) time — a 10,000× speedup on 10^4
  element inputs.

## Real World Connection

**Circular buffer monitoring in systems engineering.**

Operating systems and network devices use circular buffers
(ring buffers) to hold sensor readings, packet latencies, or
CPU utilisation samples. When an alerting system needs to
find the next spike (value exceeding a threshold) for each
sample in the ring, it must handle wrap-around — exactly
this problem.

**Other domains:**
- **Time-series forecasting:** for each hour in a 24-hour
  circular day, when is the next warmer hour?
- **Calendar scheduling:** given daily meeting lengths in a
  weekly cycle, when is the next longer meeting?
- **Game AI:** in a circular arena, each unit wants to know
  the next stronger enemy moving clockwise.

The double-pass trick with `i % n` is a clean, widely-used
technique to handle circularity without physically copying
the array, saving memory and keeping cache locality intact.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra